In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from pathlib import Path
import json
import sys
import pandas as pd

PROJECT_DIR = Path("/content/drive/MyDrive/Chunkbench")
CHUNKERS_DIR = PROJECT_DIR / "chunkers"
INPUT_DIR = PROJECT_DIR / "input"
OUTPUT_DIR = PROJECT_DIR / "output"
PREPROCESSING_DIR = PROJECT_DIR / "preprocessing"

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("Proyecto encontrado:", PROJECT_DIR.exists())
print("Ruta activa:", PROJECT_DIR)

Proyecto encontrado: True
Ruta activa: /content/drive/MyDrive/Chunkbench


In [10]:
from preprocessing.json_to_text import json_to_text

from chunkers.rulebased.fixed_character_chunking import FixedCharacterChunker
from chunkers.rulebased.fixed_token_chunking import FixedTokenChunker
from chunkers.rulebased.overlapping_token_chunking import OverlappingTokenChunker
from chunkers.rulebased.sliding_window_token_chunking import SlidingWindowTokenChunker
from chunkers.rulebased.length_aware_chunking import LengthAwareChunker
from chunkers.rulebased.sentence_based_chunking import SentenceBasedChunker
from chunkers.rulebased.sentence_group_chunking import SentenceGroupChunker
from chunkers.rulebased.paragraph_based_chunking import ParagraphBasedChunker
from chunkers.rulebased.paragraph_group_chunking import ParagraphGroupChunker

from chunkers.recursive.recursive_chunking import RecursiveChunker
from chunkers.recursive.recursive_token_chunking import RecursiveTokenChunker
from chunkers.recursive.parent_child_chunking import ParentChildChunker

from chunkers.semantic.semantic_embedding_based_chunking import SemanticEmbeddingChunker
from chunkers.semantic.semantic_similarity_threshold_chunking import SemanticSimilarityThresholdChunker
from chunkers.semantic.topic_based_chunking import TopicBasedChunker
from chunkers.semantic.semantic_boundary_detection import SemanticBoundaryChunker

from chunkers.dynamic.content_density_adaptive_chunking import ContentDensityAdaptiveChunker
from chunkers.dynamic.dynamic_token_size_chunking import DynamicTokenSizeChunker
from chunkers.dynamic.semantic_variance_adaptive_chunking import SemanticVarianceAdaptiveChunker

from chunkers.llm.llm_boundary_detection_chunking import LLMBoundaryDetectionChunker
from chunkers.llm.llm_segment_then_chunk import LLMSegmentThenChunker

In [11]:
json_path = INPUT_DIR / "EAU-Guidelines-on-Paediatric-Urology-2025.json"

with json_path.open("r", encoding="utf-8") as file:
    document = json.load(file)

document_text = json_to_text(document)

print("JSON cargado correctamente.")

JSON cargado correctamente.


## Métodos deterministas

### Fixed Character Chunking
División del texto cada número fijo de caracteres (se puede añadir overlap)

In [ ]:
fixed_character_chunking = FixedCharacterChunker(
    chunk_size=2000,
    overlap=200,
)

fixed_character_chunks = fixed_character_chunking.chunk(
    text=document_text,
    doc_id=json_path.stem,
)

print(f"Número total de chunks: {len(fixed_character_chunks)}")

for chunk in fixed_character_chunks:
    print("=" * 90)
    print(f"Chunk ID: {chunk.chunk_id}")
    print(f"Documento: {chunk.doc_id}")
    print(f"Metadatos: {chunk.metadata}")
    print("-" * 90)
    print(chunk.text)
    print()

Número total de chunks: 31
Chunk ID: EAU-Guidelines-on-Paediatric-Urology-2025_chunk_0000
Documento: EAU-Guidelines-on-Paediatric-Urology-2025
Metadatos: {'chunker': 'fixed_character_chunking', 'chunk_size': 2000, 'overlap': 200, 'start_char': 0, 'end_char': 2000, 'character_count': 2000}
------------------------------------------------------------------------------------------
Schema version: 1.0.

Document type: clinical_guideline.

Metadata. Title: EAU Guidelines on Paediatric Urology. Organization: European Association of Urology. Authors. Name: C. Radmayr. Role: Chair. Name: G. Bogaert. Role: Vice-chair. Name: A. Bujons. Name: B. Burgu. Name: M. Castagnetti. Name: L.A. ‘t Hoen. Name: F. O’Kelly. Name: N.A. Pakkasjärvi. Name: J. Quaedackers. Name: Y.F.H. Rawashdeh. Name: M.S. Silay. Name: U.K. Kennedy. Role: Guidelines Associate. Name: M. Gnech. Role: Guidelines Associate. Name: M. Skott. Role: Guidelines Associate. Name: A. van Uitert. Role: Guidelines Associate. Name: J.A. Darrau

### Fixed Token Chunking
Tokenización y división cada número fijo de tokens

In [ ]:
fixed_token_chunking = FixedTokenChunker(
    chunk_size=500,
    overlap=75,
)

fixed_token_chunks = fixed_token_chunking.chunk(
    text=document_text,
    doc_id=json_path.stem,
)

print(f"Número total de chunks: {len(fixed_token_chunks)}")

for chunk in fixed_token_chunks:
    print("=" * 90)
    print(f"Chunk ID: {chunk.chunk_id}")
    print(f"Documento: {chunk.doc_id}")
    print(f"Metadatos: {chunk.metadata}")
    print("-" * 90)
    print(chunk.text)
    print()

Número total de chunks: 30
Chunk ID: EAU-Guidelines-on-Paediatric-Urology-2025_chunk_0000
Documento: EAU-Guidelines-on-Paediatric-Urology-2025
Metadatos: {'chunker': 'fixed_token_chunking', 'chunk_size': 500, 'overlap': 75, 'start_token': 0, 'end_token': 500, 'token_count': 500}
------------------------------------------------------------------------------------------
Schema version: 1.0.

Document type: clinical_guideline.

Metadata. Title: EAU Guidelines on Paediatric Urology. Organization: European Association of Urology. Authors. Name: C. Radmayr. Role: Chair. Name: G. Bogaert. Role: Vice-chair. Name: A. Bujons. Name: B. Burgu. Name: M. Castagnetti. Name: L.A. ‘t Hoen. Name: F. O’Kelly. Name: N.A. Pakkasjärvi. Name: J. Quaedackers. Name: Y.F.H. Rawashdeh. Name: M.S. Silay. Name: U.K. Kennedy. Role: Guidelines Associate. Name: M. Gnech. Role: Guidelines Associate. Name: M. Skott. Role: Guidelines Associate. Name: A. van Uitert. Role: Guidelines Associate. Name: J.A. Darraugh. Role: 

### Overlapping Token Chunking
Tokenización y división cada número fijo de tokens con overlap

In [ ]:
overlapping_token_chunking = OverlappingTokenChunker(
    chunk_size=500,
    overlap_size=75,
)

overlapping_token_chunks = overlapping_token_chunking.chunk(
    text=document_text,
    doc_id=json_path.stem,
)

print(f"Número total de chunks: {len(overlapping_token_chunks)}")

for chunk in overlapping_token_chunks:
    print("=" * 90)
    print(f"Chunk ID: {chunk.chunk_id}")
    print(f"Documento: {chunk.doc_id}")
    print(f"Metadatos: {chunk.metadata}")
    print("-" * 90)
    print(chunk.text)
    print()

Número total de chunks: 30
Chunk ID: EAU-Guidelines-on-Paediatric-Urology-2025_chunk_0000
Documento: EAU-Guidelines-on-Paediatric-Urology-2025
Metadatos: {'chunker': 'overlapping_token_chunking', 'chunk_size': 500, 'overlap_size': 75, 'start_token': 0, 'end_token': 500, 'token_count': 500}
------------------------------------------------------------------------------------------
Schema version: 1.0.

Document type: clinical_guideline.

Metadata. Title: EAU Guidelines on Paediatric Urology. Organization: European Association of Urology. Authors. Name: C. Radmayr. Role: Chair. Name: G. Bogaert. Role: Vice-chair. Name: A. Bujons. Name: B. Burgu. Name: M. Castagnetti. Name: L.A. ‘t Hoen. Name: F. O’Kelly. Name: N.A. Pakkasjärvi. Name: J. Quaedackers. Name: Y.F.H. Rawashdeh. Name: M.S. Silay. Name: U.K. Kennedy. Role: Guidelines Associate. Name: M. Gnech. Role: Guidelines Associate. Name: M. Skott. Role: Guidelines Associate. Name: A. van Uitert. Role: Guidelines Associate. Name: J.A. Darra

### Sliding Window Token Chunking
Tokenización y desplazamiento de una ventana de tamaño fijo

In [ ]:
sliding_window_token_chunking = OverlappingTokenChunker(
    chunk_size=500,
    overlap_size=75,
)

sliding_window_token_chunks = sliding_window_token_chunking.chunk(
    text=document_text,
    doc_id=json_path.stem,
)

print(f"Número total de chunks: {len(sliding_window_token_chunks)}")

for chunk in sliding_window_token_chunks:
    print("=" * 90)
    print(f"Chunk ID: {chunk.chunk_id}")
    print(f"Documento: {chunk.doc_id}")
    print(f"Metadatos: {chunk.metadata}")
    print("-" * 90)
    print(chunk.text)
    print()

Número total de chunks: 30
Chunk ID: EAU-Guidelines-on-Paediatric-Urology-2025_chunk_0000
Documento: EAU-Guidelines-on-Paediatric-Urology-2025
Metadatos: {'chunker': 'overlapping_token_chunking', 'chunk_size': 500, 'overlap_size': 75, 'start_token': 0, 'end_token': 500, 'token_count': 500}
------------------------------------------------------------------------------------------
Schema version: 1.0.

Document type: clinical_guideline.

Metadata. Title: EAU Guidelines on Paediatric Urology. Organization: European Association of Urology. Authors. Name: C. Radmayr. Role: Chair. Name: G. Bogaert. Role: Vice-chair. Name: A. Bujons. Name: B. Burgu. Name: M. Castagnetti. Name: L.A. ‘t Hoen. Name: F. O’Kelly. Name: N.A. Pakkasjärvi. Name: J. Quaedackers. Name: Y.F.H. Rawashdeh. Name: M.S. Silay. Name: U.K. Kennedy. Role: Guidelines Associate. Name: M. Gnech. Role: Guidelines Associate. Name: M. Skott. Role: Guidelines Associate. Name: A. van Uitert. Role: Guidelines Associate. Name: J.A. Darra

### Length Aware Chunking
Separa el texto en frases y y las agrupa intentando acercarse a una longitud objetivo sin pasarla

In [ ]:
length_aware_chunking = LengthAwareChunker(
    target_length=2000,
    tolerance=75,
)

length_aware_chunks = length_aware_chunking.chunk(
    text=document_text,
    doc_id=json_path.stem,
)

print(f"Número total de chunks: {len(length_aware_chunks)}")

for chunk in length_aware_chunks:
    print("=" * 90)
    print(f"Chunk ID: {chunk.chunk_id}")
    print(f"Documento: {chunk.doc_id}")
    print(f"Metadatos: {chunk.metadata}")
    print("-" * 90)
    print(chunk.text)
    print()

Número total de chunks: 28
Chunk ID: EAU-Guidelines-on-Paediatric-Urology-2025_chunk_0000
Documento: EAU-Guidelines-on-Paediatric-Urology-2025
Metadatos: {'chunker': 'length_aware_chunking', 'target_length': 2000, 'tolerance': 75, 'min_length': 1925, 'max_length': 2075, 'length': 1569, 'sentence_count': 44}
------------------------------------------------------------------------------------------
Schema version: 1.0. Document type: clinical_guideline. Metadata. Title: EAU Guidelines on Paediatric Urology. Organization: European Association of Urology. Authors. Name: C. Radmayr. Role: Chair. Name: G. Bogaert. Role: Vice-chair. Name: A. Bujons. Name: B. Burgu. Name: M. Castagnetti. Name: L.A. ‘t Hoen. Name: F. O’Kelly. Name: N.A. Pakkasjärvi. Name: J. Quaedackers. Name: Y.F.H. Rawashdeh. Name: M.S. Silay. Name: U.K. Kennedy. Role: Guidelines Associate. Name: M. Gnech. Role: Guidelines Associate. Name: M. Skott. Role: Guidelines Associate. Name: A. van Uitert. Role: Guidelines Associate. 

### Sentence Based Chunking
Detecta las frases y convierte cada una en un chunk independiente

In [ ]:
sentence_based_chunking = SentenceBasedChunker()

sentence_based_chunks = sentence_based_chunking.chunk(
    text=document_text,
    doc_id=json_path.stem,
)

print(f"Número total de chunks: {len(sentence_based_chunks)}")

for chunk in sentence_based_chunks:
    print("=" * 90)
    print(f"Chunk ID: {chunk.chunk_id}")
    print(f"Documento: {chunk.doc_id}")
    print(f"Metadatos: {chunk.metadata}")
    print("-" * 90)
    print(chunk.text)
    print()

Se han truncado las últimas 5000 líneas del flujo de salida.
Type: regimen.

Chunk ID: EAU-Guidelines-on-Paediatric-Urology-2025_chunk_0276
Documento: EAU-Guidelines-on-Paediatric-Urology-2025
Metadatos: {'chunker': 'sentence_based_chunking', 'sentence_index': 276, 'character_count': 43}
------------------------------------------------------------------------------------------
Components: Trimethoprim, Sulfamethoxazole.

Chunk ID: EAU-Guidelines-on-Paediatric-Urology-2025_chunk_0277
Documento: EAU-Guidelines-on-Paediatric-Urology-2025
Metadatos: {'chunker': 'sentence_based_chunking', 'sentence_index': 277, 'character_count': 59}
------------------------------------------------------------------------------------------
Mechanism: Combination antibacterial agent for prophylaxis.

Chunk ID: EAU-Guidelines-on-Paediatric-Urology-2025_chunk_0278
Documento: EAU-Guidelines-on-Paediatric-Urology-2025
Metadatos: {'chunker': 'sentence_based_chunking', 'sentence_index': 278, 'character_count': 77}

### Sentence Group Chunking
Agrupa un número fijo de frases consecutivas en cada chunk, permite overlap de frases

In [ ]:
sentence_group_chunking = SentenceGroupChunker(
    sentences_per_chunk=20,
    overlap=2,
)

sentence_group_chunks = sentence_group_chunking.chunk(
    text=document_text,
    doc_id=json_path.stem,
)

print(f"Número total de chunks: {len(sentence_group_chunks)}")

for chunk in sentence_group_chunks:
    print("=" * 90)
    print(f"Chunk ID: {chunk.chunk_id}")
    print(f"Documento: {chunk.doc_id}")
    print(f"Metadatos: {chunk.metadata}")
    print("-" * 90)
    print(chunk.text)
    print()

Número total de chunks: 55
Chunk ID: EAU-Guidelines-on-Paediatric-Urology-2025_chunk_0000
Documento: EAU-Guidelines-on-Paediatric-Urology-2025
Metadatos: {'chunker': 'sentence_group_chunking', 'sentences_per_chunk': 20, 'overlap': 2, 'start_sentence_index': 0, 'end_sentence_index': 20, 'sentence_count': 20, 'character_count': 385}
------------------------------------------------------------------------------------------
Schema version: 1.0. Document type: clinical_guideline. Metadata. Title: EAU Guidelines on Paediatric Urology. Organization: European Association of Urology. Authors. Name: C. Radmayr. Role: Chair. Name: G. Bogaert. Role: Vice-chair. Name: A. Bujons. Name: B. Burgu. Name: M. Castagnetti. Name: L.A. ‘t Hoen. Name: F. O’Kelly. Name: N.A. Pakkasjärvi. Name: J. Quaedackers. Name: Y.F.H.

Chunk ID: EAU-Guidelines-on-Paediatric-Urology-2025_chunk_0001
Documento: EAU-Guidelines-on-Paediatric-Urology-2025
Metadatos: {'chunker': 'sentence_group_chunking', 'sentences_per_chunk': 

### Paragraph Based Chunking
Considera cada párrafo como un chunk independiente

In [ ]:
paragraph_based_chunking = ParagraphBasedChunker()

paragraph_based_chunks = paragraph_based_chunking.chunk(
    text=document_text,
    doc_id=json_path.stem,
)

print(f"Número total de chunks: {len(paragraph_based_chunks)}")

for chunk in paragraph_based_chunks:
    print("=" * 90)
    print(f"Chunk ID: {chunk.chunk_id}")
    print(f"Documento: {chunk.doc_id}")
    print(f"Metadatos: {chunk.metadata}")
    print("-" * 90)
    print(chunk.text)
    print()

Número total de chunks: 14
Chunk ID: EAU-Guidelines-on-Paediatric-Urology-2025_chunk_0000
Documento: EAU-Guidelines-on-Paediatric-Urology-2025
Metadatos: {'chunker': 'paragraph_based_chunking', 'paragraph_index': 0, 'character_count': 20}
------------------------------------------------------------------------------------------
Schema version: 1.0.

Chunk ID: EAU-Guidelines-on-Paediatric-Urology-2025_chunk_0001
Documento: EAU-Guidelines-on-Paediatric-Urology-2025
Metadatos: {'chunker': 'paragraph_based_chunking', 'paragraph_index': 1, 'character_count': 34}
------------------------------------------------------------------------------------------
Document type: clinical_guideline.

Chunk ID: EAU-Guidelines-on-Paediatric-Urology-2025_chunk_0002
Documento: EAU-Guidelines-on-Paediatric-Urology-2025
Metadatos: {'chunker': 'paragraph_based_chunking', 'paragraph_index': 2, 'character_count': 1506}
------------------------------------------------------------------------------------------
Meta

### Paragraph Group Chunking
Agrupa un número fijo de párrafos consecutivos en cada chunk, permite overlap de párrafos

In [ ]:
paragraph_group_chunking = ParagraphGroupChunker(
    paragraphs_per_chunk=3,
    overlap=1,
)

paragraph_group_chunks = paragraph_group_chunking.chunk(
    text=document_text,
    doc_id=json_path.stem,
)

print(f"Número total de chunks: {len(paragraph_group_chunks)}")

for chunk in paragraph_group_chunks:
    print("=" * 90)
    print(f"Chunk ID: {chunk.chunk_id}")
    print(f"Documento: {chunk.doc_id}")
    print(f"Metadatos: {chunk.metadata}")
    print("-" * 90)
    print(chunk.text)
    print()

Número total de chunks: 7
Chunk ID: EAU-Guidelines-on-Paediatric-Urology-2025_chunk_0000
Documento: EAU-Guidelines-on-Paediatric-Urology-2025
Metadatos: {'chunker': 'paragraph_group_chunking', 'paragraphs_per_chunk': 3, 'overlap': 1, 'start_paragraph_index': 0, 'end_paragraph_index': 3, 'paragraph_count': 3, 'character_count': 1564}
------------------------------------------------------------------------------------------
Schema version: 1.0.

Document type: clinical_guideline.

Metadata. Title: EAU Guidelines on Paediatric Urology. Organization: European Association of Urology. Authors. Name: C. Radmayr. Role: Chair. Name: G. Bogaert. Role: Vice-chair. Name: A. Bujons. Name: B. Burgu. Name: M. Castagnetti. Name: L.A. ‘t Hoen. Name: F. O’Kelly. Name: N.A. Pakkasjärvi. Name: J. Quaedackers. Name: Y.F.H. Rawashdeh. Name: M.S. Silay. Name: U.K. Kennedy. Role: Guidelines Associate. Name: M. Gnech. Role: Guidelines Associate. Name: M. Skott. Role: Guidelines Associate. Name: A. van Uitert. 

### Resumen

In [ ]:
chunk_results_rulebased = {
    "Fixed Character": fixed_character_chunks,
    "Fixed Token": fixed_token_chunks,
    "Length Aware": length_aware_chunks,
    "Overlapping Token": overlapping_token_chunks,
    "Paragraph Based": paragraph_based_chunks,
    "Paragraph Group": paragraph_group_chunks,
    "Sentence Based": sentence_based_chunks,
    "Sentence Group": sentence_group_chunks,
    "Sliding Window Token": sliding_window_token_chunks,
}


results_rulebased = []

for method_name, method_chunks in chunk_results_rulebased.items():
    lengths = [len(chunk.text) for chunk in method_chunks]

    results_rulebased.append(
        {
            "Método": method_name,
            "Número de chunks": len(method_chunks),
            "Longitud mínima": min(lengths) if lengths else 0,
            "Longitud máxima": max(lengths) if lengths else 0,
            "Longitud media": (
                round(sum(lengths) / len(lengths), 2)
                if lengths
                else 0
            ),
        }
    )


results_rulebased_df = pd.DataFrame(results_rulebased)

results_rulebased_df

,Método,Número de chunks,Longitud mínima,Longitud máxima,Longitud media
0,Fixed Character,31,800,2000,1961.29
1,Fixed Token,30,822,2725,2151.77
2,Length Aware,28,522,2069,1955.71
3,Overlapping Token,30,822,2725,2151.77
4,Paragraph Based,14,20,19809,3912.43
5,Paragraph Group,7,1564,33410,13506.71
6,Sentence Based,990,5,661,54.34
7,Sentence Group,55,338,2326,1106.45
8,Sliding Window Token,30,822,2725,2151.77


## Métodos recursivos

### Recursive Chunking
Divide por separadores naturales hasta cumplir un tamaño máximo de caracteres

In [ ]:
recursive_chunking = RecursiveChunker(
    chunk_size=2000,
    chunk_overlap=200,
)

recursive_chunks = recursive_chunking.chunk(
    text=document_text,
    doc_id=json_path.stem,
)

print(f"Número total de chunks: {len(recursive_chunks)}")

for chunk in recursive_chunks:
    print("=" * 90)
    print(f"Chunk ID: {chunk.chunk_id}")
    print(f"Documento: {chunk.doc_id}")
    print(f"Metadatos: {chunk.metadata}")
    print("-" * 90)
    print(chunk.text)
    print()

Número total de chunks: 35
Chunk ID: EAU-Guidelines-on-Paediatric-Urology-2025_chunk_0000
Documento: EAU-Guidelines-on-Paediatric-Urology-2025
Metadatos: {'chunker': 'recursive_chunking', 'chunk_size': 2000, 'chunk_overlap': 200, 'character_count': 1564}
------------------------------------------------------------------------------------------
Schema version: 1.0.

Document type: clinical_guideline.

Metadata. Title: EAU Guidelines on Paediatric Urology. Organization: European Association of Urology. Authors. Name: C. Radmayr. Role: Chair. Name: G. Bogaert. Role: Vice-chair. Name: A. Bujons. Name: B. Burgu. Name: M. Castagnetti. Name: L.A. ‘t Hoen. Name: F. O’Kelly. Name: N.A. Pakkasjärvi. Name: J. Quaedackers. Name: Y.F.H. Rawashdeh. Name: M.S. Silay. Name: U.K. Kennedy. Role: Guidelines Associate. Name: M. Gnech. Role: Guidelines Associate. Name: M. Skott. Role: Guidelines Associate. Name: A. van Uitert. Role: Guidelines Associate. Name: J.A. Darraugh. Role: Guidelines Office. Year: 

### Recursive Token Fallback Chunking
Divide por separadores naturales hasta cumplir un tamaño máximo de tokens

In [ ]:
recursive_token_chunking = RecursiveTokenChunker(
    chunk_size=400,
    chunk_overlap=60,
    encoding_name="cl100k_base",
)

recursive_token_chunks = recursive_token_chunking.chunk(
    text=document_text,
    doc_id=json_path.stem,
)

print(f"Número total de chunks: {len(recursive_token_chunks)}")

for chunk in recursive_token_chunks:
    print("=" * 90)
    print(f"Chunk ID: {chunk.chunk_id}")
    print(f"Documento: {chunk.doc_id}")
    print(f"Metadatos: {chunk.metadata}")
    print("-" * 90)
    print(chunk.text)
    print()

Número total de chunks: 41
Chunk ID: EAU-Guidelines-on-Paediatric-Urology-2025_chunk_0000
Documento: EAU-Guidelines-on-Paediatric-Urology-2025
Metadatos: {'chunker': 'recursive_token_fallback_chunking', 'chunk_size': 400, 'chunk_overlap': 60, 'encoding_name': 'cl100k_base', 'token_count': 15, 'character_count': 56}
------------------------------------------------------------------------------------------
Schema version: 1.0.

Document type: clinical_guideline.

Chunk ID: EAU-Guidelines-on-Paediatric-Urology-2025_chunk_0001
Documento: EAU-Guidelines-on-Paediatric-Urology-2025
Metadatos: {'chunker': 'recursive_token_fallback_chunking', 'chunk_size': 400, 'chunk_overlap': 60, 'encoding_name': 'cl100k_base', 'token_count': 398, 'character_count': 1542}
------------------------------------------------------------------------------------------
Document type: clinical_guideline.

Metadata. Title: EAU Guidelines on Paediatric Urology. Organization: European Association of Urology. Authors. Nam

### Parent Child Chunking
Crea chunks padres grandes y subdivide cada uno en chunks hijos más pequeños relacionados a los padres

In [ ]:
parent_child_chunking = ParentChildChunker(
    parent_chunk_size=2000,
    child_chunk_size=1200,
    chunk_overlap=100,
)

parent_child_chunks = parent_child_chunking.chunk(
    text=document_text,
    doc_id=json_path.stem,
)

print(f"Número total de chunks: {len(parent_child_chunks)}")

for chunk in parent_child_chunks:
    print("=" * 90)
    print(f"Chunk ID: {chunk.chunk_id}")
    print(f"Documento: {chunk.doc_id}")
    print(f"Metadatos: {chunk.metadata}")
    print("-" * 90)
    print(chunk.text)
    print()

Número total de chunks: 98
Chunk ID: EAU-Guidelines-on-Paediatric-Urology-2025_parent_0000
Documento: EAU-Guidelines-on-Paediatric-Urology-2025
Metadatos: {'chunker': 'parent_child_chunking', 'chunk_type': 'parent', 'parent_chunk_size': 2000, 'child_chunk_size': 1200, 'chunk_overlap': 100, 'character_count': 1564}
------------------------------------------------------------------------------------------
Schema version: 1.0.

Document type: clinical_guideline.

Metadata. Title: EAU Guidelines on Paediatric Urology. Organization: European Association of Urology. Authors. Name: C. Radmayr. Role: Chair. Name: G. Bogaert. Role: Vice-chair. Name: A. Bujons. Name: B. Burgu. Name: M. Castagnetti. Name: L.A. ‘t Hoen. Name: F. O’Kelly. Name: N.A. Pakkasjärvi. Name: J. Quaedackers. Name: Y.F.H. Rawashdeh. Name: M.S. Silay. Name: U.K. Kennedy. Role: Guidelines Associate. Name: M. Gnech. Role: Guidelines Associate. Name: M. Skott. Role: Guidelines Associate. Name: A. van Uitert. Role: Guidelines As

### Resumen

In [ ]:
chunk_results_recursive = {
    "Recursive": recursive_chunks,
    "Recursive Fallback Token": recursive_token_chunks,
    "Parent Child": parent_child_chunks,
}


results_recursive = []

for method_name, method_chunks in chunk_results_recursive.items():
    lengths = [len(chunk.text) for chunk in method_chunks]

    results_recursive.append(
        {
            "Método": method_name,
            "Número de chunks": len(method_chunks),
            "Longitud mínima": min(lengths) if lengths else 0,
            "Longitud máxima": max(lengths) if lengths else 0,
            "Longitud media": (
                round(sum(lengths) / len(lengths), 2)
                if lengths
                else 0
            ),
        }
    )


results_recursive_df = pd.DataFrame(results_recursive)

results_recursive_df

,Método,Número de chunks,Longitud mínima,Longitud máxima,Longitud media
0,Recursive,35,487,1997,1665.66
1,Recursive Fallback Token,41,56,2167,1474.63
2,Parent Child,98,56,1997,1153.59


## Métodos semánticos

### Semantic Embedding Based Chunking
Utiliza SemanticChunker y embeddings para detectar puntos en los que cambia el significado y agrupa las frases semánticamente relacionadas

In [ ]:
semantic_embedding_chunking = SemanticEmbeddingChunker(
    embedding_model=(
        "sentence-transformers/all-MiniLM-L6-v2"
    ),
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=95.0,
    buffer_size=1,
)

semantic_embedding_chunks = semantic_embedding_chunking.chunk(
    text=document_text,
    doc_id=json_path.stem,
)

print(
    f"Número total de chunks: "
    f"{len(semantic_embedding_chunks)}"
)

for chunk in semantic_embedding_chunks:
    print("=" * 90)
    print(f"Chunk ID: {chunk.chunk_id}")
    print(f"Documento: {chunk.doc_id}")
    print(f"Metadatos: {chunk.metadata}")
    print("-" * 90)
    print(chunk.text)
    print()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Número total de chunks: 52
Chunk ID: EAU-Guidelines-on-Paediatric-Urology-2025_chunk_0000
Documento: EAU-Guidelines-on-Paediatric-Urology-2025
Metadatos: {'chunker': 'semantic_embedding_based_chunking', 'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2', 'breakpoint_threshold_type': 'percentile', 'breakpoint_threshold_amount': 95.0, 'buffer_size': 1, 'character_count': 166}
------------------------------------------------------------------------------------------
Schema version: 1.0. Document type: clinical_guideline. Metadata. Title: EAU Guidelines on Paediatric Urology. Organization: European Association of Urology. Authors.

Chunk ID: EAU-Guidelines-on-Paediatric-Urology-2025_chunk_0001
Documento: EAU-Guidelines-on-Paediatric-Urology-2025
Metadatos: {'chunker': 'semantic_embedding_based_chunking', 'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2', 'breakpoint_threshold_type': 'percentile', 'breakpoint_threshold_amount': 95.0, 'buffer_size': 1, 'character_count': 2

### Semantic Similarity Threshold Chunking
Genera embeddings por frase y compara el de cada frase con la anterior, cambiando de chunk cuando la similitud baja

In [ ]:
semantic_similarity_threshold_chunking = (
    SemanticSimilarityThresholdChunker(
        threshold=0.50,
        embedding_model=(
            "sentence-transformers/all-MiniLM-L6-v2"
        ),
    )
)

semantic_similarity_threshold_chunks = (
    semantic_similarity_threshold_chunking.chunk(
        text=document_text,
        doc_id=json_path.stem,
    )
)

print(
    f"Número total de chunks: "
    f"{len(semantic_similarity_threshold_chunks)}"
)

for chunk in semantic_similarity_threshold_chunks:
    print("=" * 90)
    print(f"Chunk ID: {chunk.chunk_id}")
    print(f"Documento: {chunk.doc_id}")
    print(f"Metadatos: {chunk.metadata}")
    print("-" * 90)
    print(chunk.text)
    print()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Número total de chunks: 860
Chunk ID: EAU-Guidelines-on-Paediatric-Urology-2025_chunk_0000
Documento: EAU-Guidelines-on-Paediatric-Urology-2025
Metadatos: {'chunker': 'semantic_similarity_threshold_chunking', 'threshold': 0.5, 'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2', 'sentence_count': 1, 'mean_internal_similarity': None, 'character_count': 20}
------------------------------------------------------------------------------------------
Schema version: 1.0.

Chunk ID: EAU-Guidelines-on-Paediatric-Urology-2025_chunk_0001
Documento: EAU-Guidelines-on-Paediatric-Urology-2025
Metadatos: {'chunker': 'semantic_similarity_threshold_chunking', 'threshold': 0.5, 'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2', 'sentence_count': 1, 'mean_internal_similarity': None, 'character_count': 34}
------------------------------------------------------------------------------------------
Document type: clinical_guideline.

Chunk ID: EAU-Guidelines-on-Paediatric-Urology-2025_chun

### Topic Based Chunking
Genera embeddings por frase y aplica clustering aglomerativo a las frases de temática similar, manteniendo el orden original

In [ ]:
topic_based_chunking = TopicBasedChunker(
    distance_threshold=0.50,
    embedding_model=(
        "sentence-transformers/all-MiniLM-L6-v2"
    ),
)

topic_based_chunks = topic_based_chunking.chunk(
    text=document_text,
    doc_id=json_path.stem,
)

print(
    f"Número total de chunks: "
    f"{len(topic_based_chunks)}"
)

for chunk in topic_based_chunks:
    print("=" * 90)
    print(f"Chunk ID: {chunk.chunk_id}")
    print(f"Documento: {chunk.doc_id}")
    print(f"Metadatos: {chunk.metadata}")
    print("-" * 90)
    print(chunk.text)
    print()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Número total de chunks: 895
Chunk ID: EAU-Guidelines-on-Paediatric-Urology-2025_chunk_0000
Documento: EAU-Guidelines-on-Paediatric-Urology-2025
Metadatos: {'chunker': 'topic_based_chunking', 'topic_label': 115, 'sentence_count': 1}
------------------------------------------------------------------------------------------
Schema version: 1.0.

Chunk ID: EAU-Guidelines-on-Paediatric-Urology-2025_chunk_0001
Documento: EAU-Guidelines-on-Paediatric-Urology-2025
Metadatos: {'chunker': 'topic_based_chunking', 'topic_label': 125, 'sentence_count': 1}
------------------------------------------------------------------------------------------
Document type: clinical_guideline.

Chunk ID: EAU-Guidelines-on-Paediatric-Urology-2025_chunk_0002
Documento: EAU-Guidelines-on-Paediatric-Urology-2025
Metadatos: {'chunker': 'topic_based_chunking', 'topic_label': 36, 'sentence_count': 1}
------------------------------------------------------------------------------------------
Metadata.

Chunk ID: EAU-Guide

### Semantic Boundary Detection Chunking
Utiliza SemanticChunker y separa los chunks cuando hay cambios bruscos en la evolución de las distancias semánticas

In [ ]:
semantic_boundary_chunking = SemanticBoundaryChunker(
    embedding_model=(
        "sentence-transformers/all-MiniLM-L6-v2"
    ),
    breakpoint_threshold_amount=95.0,
    buffer_size=1,
)

semantic_boundary_chunks = semantic_boundary_chunking.chunk(
    text=document_text,
    doc_id=json_path.stem,
)

print(
    f"Número total de chunks: "
    f"{len(semantic_boundary_chunks)}"
)

for chunk in semantic_boundary_chunks:
    print("=" * 90)
    print(f"Chunk ID: {chunk.chunk_id}")
    print(f"Documento: {chunk.doc_id}")
    print(f"Metadatos: {chunk.metadata}")
    print("-" * 90)
    print(chunk.text)
    print()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Número total de chunks: 52
Chunk ID: EAU-Guidelines-on-Paediatric-Urology-2025_chunk_0000
Documento: EAU-Guidelines-on-Paediatric-Urology-2025
Metadatos: {'chunker': 'semantic_boundary_detection', 'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2', 'breakpoint_threshold_type': 'gradient', 'breakpoint_threshold_amount': 95.0, 'buffer_size': 1, 'character_count': 20}
------------------------------------------------------------------------------------------
Schema version: 1.0.

Chunk ID: EAU-Guidelines-on-Paediatric-Urology-2025_chunk_0001
Documento: EAU-Guidelines-on-Paediatric-Urology-2025
Metadatos: {'chunker': 'semantic_boundary_detection', 'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2', 'breakpoint_threshold_type': 'gradient', 'breakpoint_threshold_amount': 95.0, 'buffer_size': 1, 'character_count': 136}
------------------------------------------------------------------------------------------
Document type: clinical_guideline. Metadata. Title: EAU Guidelines o

### Resumen

In [ ]:
chunk_results_semantic = {
    "Semantic Boundary Detection": semantic_boundary_chunks,
    "Semantic Embedding Based": semantic_embedding_chunks,
    "Semantic Similarity Threshold": semantic_similarity_threshold_chunks,
    "Topic Based": topic_based_chunks,
}


results_semantic = []

for method_name, method_chunks in chunk_results_semantic.items():
    lengths = [len(chunk.text) for chunk in method_chunks]

    results_semantic.append(
        {
            "Método": method_name,
            "Número de chunks": len(method_chunks),
            "Longitud mínima": min(lengths) if lengths else 0,
            "Longitud máxima": max(lengths) if lengths else 0,
            "Longitud media": (
                round(sum(lengths) / len(lengths), 2)
                if lengths
                else 0
            ),
        }
    )


results_semantic_df = pd.DataFrame(results_semantic)

results_semantic_df

,Método,Número de chunks,Longitud mínima,Longitud máxima,Longitud media
0,Semantic Boundary Detection,52,5,10304,1052.62
1,Semantic Embedding Based,52,20,11103,1052.62
2,Semantic Similarity Threshold,860,5,954,62.71
3,Topic Based,895,5,822,60.22


## Métodos dinámicos

### Content Density Adaptative Chunking
Divide en frases y agrupa según la densidad léxica (a mayor densidad mayor tamaño)

In [ ]:
content_density_adaptive_chunking = (
    ContentDensityAdaptiveChunker(
        base_chunk_size=2000,
    )
)

content_density_adaptive_chunks = (
    content_density_adaptive_chunking.chunk(
        text=document_text,
        doc_id=json_path.stem,
    )
)

print(
    f"Número total de chunks: "
    f"{len(content_density_adaptive_chunks)}"
)

for chunk in content_density_adaptive_chunks:
    print("=" * 90)
    print(f"Chunk ID: {chunk.chunk_id}")
    print(f"Documento: {chunk.doc_id}")
    print(f"Metadatos: {chunk.metadata}")
    print("-" * 90)
    print(chunk.text)
    print()

Número total de chunks: 55
Chunk ID: EAU-Guidelines-on-Paediatric-Urology-2025_chunk_0000
Documento: EAU-Guidelines-on-Paediatric-Urology-2025
Metadatos: {'chunker': 'content_density_adaptive_chunking', 'base_chunk_size': 2000, 'mean_density': 1.0, 'mean_density_factor': 0.5, 'sentence_count': 42, 'character_count': 959}
------------------------------------------------------------------------------------------
Schema version: 1.0. Document type: clinical_guideline. Metadata. Title: EAU Guidelines on Paediatric Urology. Organization: European Association of Urology. Authors. Name: C. Radmayr. Role: Chair. Name: G. Bogaert. Role: Vice-chair. Name: A. Bujons. Name: B. Burgu. Name: M. Castagnetti. Name: L.A. ‘t Hoen. Name: F. O’Kelly. Name: N.A. Pakkasjärvi. Name: J. Quaedackers. Name: Y.F.H. Rawashdeh. Name: M.S. Silay. Name: U.K. Kennedy. Role: Guidelines Associate. Name: M. Gnech. Role: Guidelines Associate. Name: M. Skott. Role: Guidelines Associate. Name: A. van Uitert. Role: Guidelin

### Dynamic Token Size Chunking
Tokeniza y crea chunks de tamaño variable intentando separar por signos de puntuación o salto de línea

In [ ]:
dynamic_token_size_chunking = DynamicTokenSizeChunker(
    min_chunk_size=250,
    max_chunk_size=500,
    encoding_name="cl100k_base",
)

dynamic_token_size_chunks = (
    dynamic_token_size_chunking.chunk(
        text=document_text,
        doc_id=json_path.stem,
    )
)

print(
    f"Número total de chunks: "
    f"{len(dynamic_token_size_chunks)}"
)

for chunk in dynamic_token_size_chunks:
    print("=" * 90)
    print(f"Chunk ID: {chunk.chunk_id}")
    print(f"Documento: {chunk.doc_id}")
    print(f"Metadatos: {chunk.metadata}")
    print("-" * 90)
    print(chunk.text)
    print()

Número total de chunks: 26
Chunk ID: EAU-Guidelines-on-Paediatric-Urology-2025_chunk_0000
Documento: EAU-Guidelines-on-Paediatric-Urology-2025
Metadatos: {'chunker': 'dynamic_token_size_chunking', 'min_chunk_size': 250, 'max_chunk_size': 500, 'encoding_name': 'cl100k_base', 'start_token': 0, 'end_token': 408, 'token_count': 408, 'character_count': 1572, 'split_reason': 'sentence_end'}
------------------------------------------------------------------------------------------
Schema version: 1.0.

Document type: clinical_guideline.

Metadata. Title: EAU Guidelines on Paediatric Urology. Organization: European Association of Urology. Authors. Name: C. Radmayr. Role: Chair. Name: G. Bogaert. Role: Vice-chair. Name: A. Bujons. Name: B. Burgu. Name: M. Castagnetti. Name: L.A. ‘t Hoen. Name: F. O’Kelly. Name: N.A. Pakkasjärvi. Name: J. Quaedackers. Name: Y.F.H. Rawashdeh. Name: M.S. Silay. Name: U.K. Kennedy. Role: Guidelines Associate. Name: M. Gnech. Role: Guidelines Associate. Name: M. Sko

### Semantic Variance Adaptative Chunking
Genera embeddings, calcula la similitud entre frases y corta el chunk cuando la similitud cae respecto a la media

In [ ]:
semantic_variance_adaptive_chunking = (
    SemanticVarianceAdaptiveChunker(
        sensitivity=0.15,
        embedding_model=(
            "sentence-transformers/all-MiniLM-L6-v2"
        ),
        window_size=3,
    )
)

semantic_variance_adaptive_chunks = (
    semantic_variance_adaptive_chunking.chunk(
        text=document_text,
        doc_id=json_path.stem,
    )
)

print(
    f"Número total de chunks: "
    f"{len(semantic_variance_adaptive_chunks)}"
)

for chunk in semantic_variance_adaptive_chunks:
    print("=" * 90)
    print(f"Chunk ID: {chunk.chunk_id}")
    print(f"Documento: {chunk.doc_id}")
    print(f"Metadatos: {chunk.metadata}")
    print("-" * 90)
    print(chunk.text)
    print()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Número total de chunks: 204
Chunk ID: EAU-Guidelines-on-Paediatric-Urology-2025_chunk_0000
Documento: EAU-Guidelines-on-Paediatric-Urology-2025
Metadatos: {'chunker': 'semantic_variance_adaptive_chunking', 'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2', 'sensitivity': 0.15, 'window_size': 3, 'split_similarity': 0.0407, 'recent_mean_similarity': 0.2372, 'mean_internal_similarity': 0.2372, 'start_sentence_index': 0, 'end_sentence_index': 3, 'sentence_count': 3, 'character_count': 65}
------------------------------------------------------------------------------------------
Schema version: 1.0. Document type: clinical_guideline. Metadata.

Chunk ID: EAU-Guidelines-on-Paediatric-Urology-2025_chunk_0001
Documento: EAU-Guidelines-on-Paediatric-Urology-2025
Metadatos: {'chunker': 'semantic_variance_adaptive_chunking', 'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2', 'sensitivity': 0.15, 'window_size': 3, 'split_similarity': 0.12, 'recent_mean_similarity': 0.2766, 'mea

### Resumen

In [ ]:
chunk_results_dynamic = {
    "Content Density Adaptative": content_density_adaptive_chunks,
    "Dynamic Token Size": dynamic_token_size_chunks,
    "Semantic Variance Adaptative": semantic_variance_adaptive_chunks,
}


results_dynamic = []

for method_name, method_chunks in chunk_results_dynamic.items():
    lengths = [len(chunk.text) for chunk in method_chunks]

    results_dynamic.append(
        {
            "Método": method_name,
            "Número de chunks": len(method_chunks),
            "Longitud mínima": min(lengths) if lengths else 0,
            "Longitud máxima": max(lengths) if lengths else 0,
            "Longitud media": (
                round(sum(lengths) / len(lengths), 2)
                if lengths
                else 0
            ),
        }
    )


results_dynamic_df = pd.DataFrame(results_dynamic)

results_dynamic_df

,Método,Número de chunks,Longitud mínima,Longitud máxima,Longitud media
0,Content Density Adaptative,55,771,1279,995.15
1,Dynamic Token Size,26,1136,2619,2106.73
2,Semantic Variance Adaptative,204,6,4578,267.57


## Métodos basados en LLM

### LLM Boundary Detection Chunking
Genera chunks provisionales con reglas simples y si detecta tamaños desiguales, un LLM reajusta las posiciones de corte. Añade overlap

In [ ]:
llm_boundary_detection_chunking = (
    LLMBoundaryDetectionChunker(
        max_chars=2000,
        overlap=200,
        enable_llm_refinement=True,
        llm_timeout_sec=15.0,
        llm_max_tokens=256,
        llm_temperature=0.0,
    )
)

llm_boundary_detection_chunks = (
    llm_boundary_detection_chunking.chunk(
        text=document_text,
        doc_id=json_path.stem,
    )
)

print(
    f"Número total de chunks: "
    f"{len(llm_boundary_detection_chunks)}"
)

for chunk in llm_boundary_detection_chunks:
    print("=" * 90)
    print(f"Chunk ID: {chunk.chunk_id}")
    print(f"Documento: {chunk.doc_id}")
    print(f"Metadatos: {chunk.metadata}")
    print("-" * 90)
    print(chunk.text)
    print()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] If you want to use `BertLMHeadModel` as a standalone, add `is_decoder=True.`


Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

[transformers] This checkpoint seem corrupted. The tied weights mapping for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are absent from the checkpoint, and we could not find another related tied weight for those keys
[transformers] BertLMHeadModel LOAD REPORT from: ./models/sentence-transformers__all-MiniLM-L6-v2
Key                                        | Status     | 
-------------------------------------------+------------+-
pooler.dense.weight                        | UNEXPECTED | 
pooler.dense.bias                          | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | MISSING    | 
cls.predictions.bias                       | MISSING    | 
cls.predictions.transform.dense.bias       | MISSING    | 
cls.predictions.transform.LayerNorm.weight | MISSING    | 
cls.predictions.transform.dense.weight     | MISSING    | 
cls.predictions.decoder.bias               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loadin

Número total de chunks: 29
Chunk ID: EAU-Guidelines-on-Paediatric-Urology-2025_chunk_0000
Documento: EAU-Guidelines-on-Paediatric-Urology-2025
Metadatos: {'chunker': 'llm_boundary_detection_chunking', 'max_chars': 2000, 'overlap': 200, 'character_count': 1569, 'used_llm_refinement': False, 'chunk_index': 0}
------------------------------------------------------------------------------------------
Schema version: 1.0. Document type: clinical_guideline. Metadata. Title: EAU Guidelines on Paediatric Urology. Organization: European Association of Urology. Authors. Name: C. Radmayr. Role: Chair. Name: G. Bogaert. Role: Vice-chair. Name: A. Bujons. Name: B. Burgu. Name: M. Castagnetti. Name: L.A. ‘t Hoen. Name: F. O’Kelly. Name: N.A. Pakkasjärvi. Name: J. Quaedackers. Name: Y.F.H. Rawashdeh. Name: M.S. Silay. Name: U.K. Kennedy. Role: Guidelines Associate. Name: M. Gnech. Role: Guidelines Associate. Name: M. Skott. Role: Guidelines Associate. Name: A. van Uitert. Role: Guidelines Associate. 

### LLM Segment Then Chunk
Segmenta el documento por estructura y envía al LLM los segmentos grandes o ambiguos para subdividirlos

In [ ]:
llm_segment_then_chunking = LLMSegmentThenChunker(
    max_chars=2000,
    overlap=200,
    llm_refine_threshold=1200,
    enable_llm_refinement=True,
    llm_timeout_sec=15.0,
    llm_max_tokens=256,
    llm_temperature=0.0,
)

llm_segment_then_chunks = (
    llm_segment_then_chunking.chunk(
        text=document_text,
        doc_id=json_path.stem,
    )
)

print(
    f"Número total de chunks: "
    f"{len(llm_segment_then_chunks)}"
)

for chunk in llm_segment_then_chunks:
    print("=" * 90)
    print(f"Chunk ID: {chunk.chunk_id}")
    print(f"Documento: {chunk.doc_id}")
    print(f"Metadatos: {chunk.metadata}")
    print("-" * 90)
    print(chunk.text)
    print()

[transformers] If you want to use `BertLMHeadModel` as a standalone, add `is_decoder=True.`


Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

[transformers] This checkpoint seem corrupted. The tied weights mapping for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are absent from the checkpoint, and we could not find another related tied weight for those keys
[transformers] BertLMHeadModel LOAD REPORT from: ./models/sentence-transformers__all-MiniLM-L6-v2
Key                                        | Status     | 
-------------------------------------------+------------+-
pooler.dense.weight                        | UNEXPECTED | 
pooler.dense.bias                          | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | MISSING    | 
cls.predictions.bias                       | MISSING    | 
cls.predictions.transform.dense.bias       | MISSING    | 
cls.predictions.transform.LayerNorm.weight | MISSING    | 
cls.predictions.transform.dense.weight     | MISSING    | 
cls.predictions.decoder.bias               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loadin

Número total de chunks: 37
Chunk ID: EAU-Guidelines-on-Paediatric-Urology-2025_chunk_0000
Documento: EAU-Guidelines-on-Paediatric-Urology-2025
Metadatos: {'chunker': 'llm_segment_then_chunking', 'max_chars': 2000, 'overlap': 200, 'llm_refine_threshold': 1200, 'unit_type': 'structured_paragraph', 'source_segment_index': 0, 'segment_part_index': 0, 'used_llm_refinement': False, 'chunk_index': 0, 'character_count': 20}
------------------------------------------------------------------------------------------
Schema version: 1.0.

Chunk ID: EAU-Guidelines-on-Paediatric-Urology-2025_chunk_0001
Documento: EAU-Guidelines-on-Paediatric-Urology-2025
Metadatos: {'chunker': 'llm_segment_then_chunking', 'max_chars': 2000, 'overlap': 200, 'llm_refine_threshold': 1200, 'unit_type': 'structured_paragraph', 'source_segment_index': 1, 'segment_part_index': 0, 'used_llm_refinement': False, 'chunk_index': 1, 'character_count': 56}
--------------------------------------------------------------------------

### Resumen

In [ ]:
chunk_results_llm = {
    "LLM Boundary Detection": llm_boundary_detection_chunks,
    "LLM Segment Then Chunk": llm_segment_then_chunks,
}


results_llm = []

for method_name, method_chunks in chunk_results_llm.items():
    lengths = [len(chunk.text) for chunk in method_chunks]

    results_llm.append(
        {
            "Método": method_name,
            "Número de chunks": len(method_chunks),
            "Longitud mínima": min(lengths) if lengths else 0,
            "Longitud máxima": max(lengths) if lengths else 0,
            "Longitud media": (
                round(sum(lengths) / len(lengths), 2)
                if lengths
                else 0
            ),
        }
    )


results_llm_df = pd.DataFrame(results_llm)

results_llm_df

,Método,Número de chunks,Longitud mínima,Longitud máxima,Longitud media
0,LLM Boundary Detection,29,952,2196,2076.48
1,LLM Segment Then Chunk,37,20,2200,1659.19


## Resumen global

In [34]:
import pandas as pd
import numpy as np


# Separamos parent y child para no mezclar
# dos niveles distintos de chunking
parent_chunks = [
    chunk
    for chunk in parent_child_chunks
    if chunk.metadata.get("chunk_type") == "parent"
]

child_chunks = [
    chunk
    for chunk in parent_child_chunks
    if chunk.metadata.get("chunk_type") == "child"
]


chunk_results = {
    "Fixed Character": fixed_character_chunks,
    "Fixed Token": fixed_token_chunks,
    "Length Aware": length_aware_chunks,
    "Overlapping Token": overlapping_token_chunks,
    "Paragraph Based": paragraph_based_chunks,
    "Paragraph Group": paragraph_group_chunks,
    "Sentence Based": sentence_based_chunks,
    "Sentence Group": sentence_group_chunks,
    "Sliding Window Token": sliding_window_token_chunks,
    "Recursive": recursive_chunks,
    "Recursive Fallback Token": recursive_token_chunks,
    "Parent Child - Parent": parent_chunks,
    "Parent Child - Child": child_chunks,
    "Semantic Boundary Detection": semantic_boundary_chunks,
    "Semantic Embedding Based": semantic_embedding_chunks,
    "Semantic Similarity Threshold": (
        semantic_similarity_threshold_chunks
    ),
    "Topic Based": topic_based_chunks,
    "Content Density Adaptive": (
        content_density_adaptive_chunks
    ),
    "Dynamic Token Size": dynamic_token_size_chunks,
    "Semantic Variance Adaptive": (
        semantic_variance_adaptive_chunks
    ),
    "LLM Boundary Detection": (
        llm_boundary_detection_chunks
    ),
    "LLM Segment Then Chunk": (
        llm_segment_then_chunks
    ),
}


original_character_count = len(document_text)

results = []


for method_name, method_chunks in chunk_results.items():

    texts = [
        chunk.text.strip()
        for chunk in method_chunks
        if chunk.text and chunk.text.strip()
    ]

    lengths = [
        len(text)
        for text in texts
    ]

    if not lengths:
        results.append(
            {
                "Método": method_name,
                "Número de chunks": 0,
                "Caracteres mínimos": 0,
                "Caracteres P25": 0,
                "Caracteres mediana": 0,
                "Caracteres medios": 0,
                "Caracteres P75": 0,
                "Caracteres máximos": 0,
                "Desviación estándar": 0,
                "Chunks < 400 caracteres (%)": 0,
                "Caracteres totales": 0,
                "Ratio caracteres/original": 0,
            }
        )
        continue

    small_chunks = sum(
        length < 400
        for length in lengths
    )

    total_characters = sum(lengths)

    results.append(
        {
            "Método": method_name,
            "Número de chunks": len(lengths),

            "Caracteres mínimos": min(lengths),

            "Caracteres P25": round(
                np.percentile(lengths, 25),
                2,
            ),

            "Caracteres mediana": round(
                np.median(lengths),
                2,
            ),

            "Caracteres medios": round(
                np.mean(lengths),
                2,
            ),

            "Caracteres P75": round(
                np.percentile(lengths, 75),
                2,
            ),

            "Caracteres máximos": max(lengths),

            "Desviación estándar": round(
                np.std(lengths),
                2,
            ),

            "Chunks < 400 caracteres (%)": round(
                small_chunks / len(lengths) * 100,
                2,
            ),

            "Caracteres totales": total_characters,

            "Ratio caracteres/original": round(
                total_characters
                / original_character_count,
                3,
            )
            if original_character_count > 0
            else 0,
        }
    )


results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="Caracteres mediana",
    ascending=True,
).reset_index(drop=True)

results_df

,Método,Número de chunks,Caracteres mínimos,Caracteres P25,Caracteres mediana,Caracteres medios,Caracteres P75,Caracteres máximos,Desviación estándar,Chunks < 400 caracteres (%),Caracteres totales,Ratio caracteres/original
0,Semantic Similarity Threshold,860,5,15.00,21.0,62.71,66.75,954,98.45,98.60,53928,0.984
1,Topic Based,895,5,15.00,21.0,60.22,69.00,822,87.68,98.99,53893,0.983
2,Sentence Based,990,5,16.00,25.0,54.34,80.75,661,60.88,99.70,53798,0.982
3,Semantic Variance Adaptive,204,6,44.00,158.0,267.57,320.75,4578,413.42,78.43,54584,0.996
4,Semantic Boundary Detection,52,5,19.25,262.0,1052.62,940.00,10304,1962.58,55.77,54736,0.999
5,Semantic Embedding Based,52,20,33.00,270.5,1052.62,1029.25,11103,1989.14,51.92,54736,0.999
6,Parent Child - Child,63,56,774.50,917.0,906.95,1158.00,1199,278.93,6.35,57138,1.043
7,Content Density Adaptive,55,771,955.00,987.0,995.15,1009.00,1279,89.34,0.00,54733,0.999
8,Sentence Group,55,338,396.50,1091.0,1106.45,1528.00,2326,604.68,25.45,60855,1.110
9,Recursive Fallback Token,41,56,1009.00,1692.0,1474.63,1801.00,2167,506.62,2.44,60460,1.103


In [32]:
import pandas as pd
import numpy as np
import tiktoken


# Tokenizador común para comparar todos los métodos
encoding = tiktoken.get_encoding("cl100k_base")


# Separamos parent y child para no mezclar dos niveles distintos
parent_chunks = [
    chunk
    for chunk in parent_child_chunks
    if chunk.metadata.get("chunk_type") == "parent"
]

child_chunks = [
    chunk
    for chunk in parent_child_chunks
    if chunk.metadata.get("chunk_type") == "child"
]


chunk_results = {
    "Fixed Character": fixed_character_chunks,
    "Fixed Token": fixed_token_chunks,
    "Length Aware": length_aware_chunks,
    "Overlapping Token": overlapping_token_chunks,
    "Paragraph Based": paragraph_based_chunks,
    "Paragraph Group": paragraph_group_chunks,
    "Sentence Based": sentence_based_chunks,
    "Sentence Group": sentence_group_chunks,
    "Sliding Window Token": sliding_window_token_chunks,
    "Recursive": recursive_chunks,
    "Recursive Fallback Token": recursive_token_chunks,
    "Parent Child - Parent": parent_chunks,
    "Parent Child - Child": child_chunks,
    "Semantic Boundary Detection": semantic_boundary_chunks,
    "Semantic Embedding Based": semantic_embedding_chunks,
    "Semantic Similarity Threshold": semantic_similarity_threshold_chunks,
    "Topic Based": topic_based_chunks,
    "Content Density Adaptive": content_density_adaptive_chunks,
    "Dynamic Token Size": dynamic_token_size_chunks,
    "Semantic Variance Adaptive": semantic_variance_adaptive_chunks,
    "LLM Boundary Detection": llm_boundary_detection_chunks,
    "LLM Segment Then Chunk": llm_segment_then_chunks,
}


original_token_count = len(
    encoding.encode(document_text)
)

results = []


for method_name, method_chunks in chunk_results.items():

    # Eliminamos posibles chunks vacíos
    texts = [
        chunk.text.strip()
        for chunk in method_chunks
        if chunk.text and chunk.text.strip()
    ]

    character_lengths = [
        len(text)
        for text in texts
    ]

    token_lengths = [
        len(encoding.encode(text))
        for text in texts
    ]

    if not texts:
        results.append(
            {
                "Método": method_name,
                "Número de chunks": 0,
            }
        )
        continue

    total_tokens = sum(token_lengths)

    small_chunks = sum(
        token_count < 100
        for token_count in token_lengths
    )

    results.append(
        {
            "Método": method_name,
            "Número de chunks": len(texts),

            "Caracteres mínimos": min(
                character_lengths
            ),
            "Caracteres máximos": max(
                character_lengths
            ),
            "Caracteres medios": round(
                np.mean(character_lengths),
                2,
            ),

            "Tokens mínimos": min(
                token_lengths
            ),
            "Tokens P25": round(
                np.percentile(token_lengths, 25),
                2,
            ),
            "Tokens mediana": round(
                np.median(token_lengths),
                2,
            ),
            "Tokens medios": round(
                np.mean(token_lengths),
                2,
            ),
            "Tokens P75": round(
                np.percentile(token_lengths, 75),
                2,
            ),
            "Tokens máximos": max(
                token_lengths
            ),
            "Desviación tokens": round(
                np.std(token_lengths),
                2,
            ),

            "Chunks < 100 tokens (%)": round(
                small_chunks / len(texts) * 100,
                2,
            ),

            "Tokens totales": total_tokens,

            "Ratio tokens/original": round(
                total_tokens / original_token_count,
                3,
            )
            if original_token_count > 0
            else 0,
        }
    )


results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="Tokens mediana",
    ascending=True,
).reset_index(drop=True)

results_df

,Método,Número de chunks,Caracteres mínimos,Caracteres máximos,Caracteres medios,Tokens mínimos,Tokens P25,Tokens mediana,Tokens medios,Tokens P75,Tokens máximos,Desviación tokens,Chunks < 100 tokens (%),Tokens totales,Ratio tokens/original
0,Semantic Similarity Threshold,860,5,954,62.71,2,6.00,7.0,14.81,14.25,199,20.05,98.84,12740,1.014
1,Topic Based,895,5,822,60.22,2,6.00,7.0,14.24,15.00,212,17.98,99.11,12745,1.014
2,Sentence Based,990,5,661,54.34,2,7.00,8.0,12.88,17.00,142,12.28,99.60,12754,1.015
3,Semantic Variance Adaptive,204,6,4578,267.57,2,11.00,36.0,61.63,67.00,1730,131.74,81.86,12572,1.000
4,Semantic Boundary Detection,52,5,10304,1052.62,2,7.00,62.5,241.67,207.75,2162,453.54,57.69,12567,1.000
5,Semantic Embedding Based,52,20,11103,1052.62,7,12.00,65.0,241.63,218.25,2344,459.54,51.92,12565,1.000
6,Parent Child - Child,63,56,1199,906.95,15,165.50,210.0,209.29,252.00,485,82.33,6.35,13185,1.049
7,Content Density Adaptive,55,771,1279,995.15,157,198.00,216.0,228.56,230.00,408,52.47,0.00,12571,1.000
8,Sentence Group,55,338,2326,1106.45,107,149.00,238.0,254.40,321.00,524,108.41,0.00,13992,1.113
9,Recursive Fallback Token,41,56,2167,1474.63,15,372.00,384.0,339.61,393.00,399,99.27,2.44,13924,1.108
